# EDA — Precipitação e Relação Chuva-Vazão
## Bacia do Itajaí-Açu | INMET + CHIRPS

---

### O que este notebook responde

1. As estações INMET são representativas da bacia inteira ou são pontuais demais?
2. O CHIRPS concorda com as estações INMET? Onde diverge?
3. **Qual é o lag entre pico de chuva e pico de vazão em Blumenau?**
   ← Esta resposta define o `hindcast_length` no YAML de treinamento.
4. A relação chuva-vazão varia por época do ano (solo saturado vs. seco)?

### Por que fazer isso antes de treinar?

O `hindcast_length` (janela de contexto histórico do LSTM) deve ser
pelo menos tão longo quanto o lag chuva-vazão característico da bacia.
Para o Itajaí-Açu, com ~15.000 km², o tempo de concentração estimado
é de 24–48 h para sub-bacias e até 72 h para a bacia completa acima de
Blumenau. Se o config atual usa 168 h (7 dias), isso deve ser suficiente
— mas vamos confirmar empiricamente.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '../../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from scipy import signal, stats

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

FIGDIR = Path('../../reports/figures')
FIGDIR.mkdir(parents=True, exist_ok=True)

RAW_DIR    = Path('../../data/raw')
PRECIP_DIR = RAW_DIR / 'precipitation'
FLOW_DIR   = RAW_DIR / 'streamflow'

STATION_ANA = '83500000'

FLOOD_EVENTS = {
    '1983-11-09': 'Nov/1983',
    '2008-11-23': 'Nov/2008',
    '2011-09-08': 'Set/2011',
}

print('Ambiente configurado.')

## 1. Download e Carregamento dos Dados

In [ ]:
from src.data.ana_downloader import download_series, save_raw
from src.data.precipitation_downloader import (
    download_all_basin_stations,
    download_chirps_range,
    chirps_basin_average,
    list_inmet_stations_in_basin,
)

# ── Vazão ─────────────────────────────────────────────────────────────────────
vazao_path = FLOW_DIR / f'{STATION_ANA}_vazao_raw.parquet'
if not vazao_path.exists():
    df_q = download_series(STATION_ANA, 'vazao', start_year=1981)
    save_raw(df_q, STATION_ANA, 'vazao')
else:
    df_q = pd.read_parquet(vazao_path)

Q = df_q['value'].rename('Q_m3s')

# ── Precipitação INMET ────────────────────────────────────────────────────────
inmet_files = list(PRECIP_DIR.glob('inmet_*_daily.parquet'))
if not inmet_files:
    print('Baixando estações INMET da bacia...')
    inmet_data = download_all_basin_stations(start_date='2000-01-01', out_dir=PRECIP_DIR)
    inmet_files = list(PRECIP_DIR.glob('inmet_*_daily.parquet'))
else:
    print(f'{len(inmet_files)} estação(ões) INMET já baixadas.')

# Carregar todas as estações INMET em um único DataFrame
inmet_dfs = {}
for f in inmet_files:
    sid = f.stem.replace('inmet_', '').replace('_daily', '')
    inmet_dfs[sid] = pd.read_parquet(f)['precip_mm']

if inmet_dfs:
    df_inmet = pd.DataFrame(inmet_dfs)
    df_inmet.index = pd.to_datetime(df_inmet.index)
    print(f'INMET: {len(df_inmet.columns)} estações, {df_inmet.index.min().date()} → {df_inmet.index.max().date()}')
else:
    df_inmet = pd.DataFrame()
    print('Nenhuma estação INMET disponível localmente.')

# ── Precipitação CHIRPS ───────────────────────────────────────────────────────
chirps_files = list(PRECIP_DIR.glob('chirps_*_itajai.nc'))
if not chirps_files:
    print('Baixando CHIRPS... (pode demorar alguns minutos)')
    chirps_files = download_chirps_range(start_year=1981, out_dir=PRECIP_DIR)
else:
    print(f'{len(chirps_files)} arquivo(s) CHIRPS já baixados.')

if chirps_files:
    df_chirps = chirps_basin_average(PRECIP_DIR)
    print(f'CHIRPS: {df_chirps.index.min().date()} → {df_chirps.index.max().date()}')
else:
    df_chirps = pd.DataFrame()
    print('Nenhum dado CHIRPS disponível.')

print('\nDados carregados.')

## 2. Mapa das Estações INMET na Bacia

Visualizar a distribuição espacial antes de qualquer análise.

**O que procurar:** estações cobrindo as cabeceiras dos três tributários
principais (Itajaí do Sul, Itajaí do Oeste, Itajaí do Norte) e a região
costeira próxima a Blumenau. Estações concentradas apenas na foz
sub-representam a chuva das cabeceiras — que é onde os grandes eventos
se iniciam (chuva orográfica na Serra).

In [ ]:
try:
    stations = list_inmet_stations_in_basin()

    fig, ax = plt.subplots(figsize=(9, 7))

    if not stations.empty:
        sc = ax.scatter(
            stations['lon'], stations['lat'],
            c='steelblue', s=80, zorder=5, label='Estações INMET automáticas'
        )
        for _, row in stations.iterrows():
            ax.annotate(
                row.get('station_id', ''),
                (row['lon'], row['lat']),
                textcoords='offset points', xytext=(5, 3),
                fontsize=7
            )

    # Marcar estação fluviométrica de Blumenau
    ax.scatter(-49.065, -26.915, c='crimson', s=120, marker='v', zorder=6,
               label='Estação ANA 83500000 (Blumenau)')

    # Bounding box da bacia
    from matplotlib.patches import Rectangle
    ax.add_patch(Rectangle(
        (-50.2, -27.6), 50.2 - 48.6, 27.6 - 26.5,
        fill=False, edgecolor='gray', lw=1.5, ls='--', label='Bounding box da bacia'
    ))

    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Estações INMET — Bacia do Itajaí-Açu', fontsize=12)
    ax.legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(FIGDIR / '09_mapa_estacoes_inmet.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Estações na bacia: {len(stations)}')
    if not stations.empty:
        print(stations[['station_id', 'name', 'lat', 'lon', 'start_date']].to_string())

except Exception as e:
    print(f'API INMET indisponível: {e}')
    print('Prosseguindo com análise CHIRPS.')

## 3. Completude das Estações INMET

Mesma análise de heatmap do notebook de vazão, mas para as estações de chuva.

**Resultado esperado e possível problema:**
Estações automáticas INMET têm cobertura desde ~2000, mas com gaps
frequentes (manutenção de sensor, falha de transmissão, baterias).
Gaps nas estações de chuva são mais problemáticos que na vazão porque
a chuva é mais intermitente — um dia de NaN pode ser ausência de dado
ou dia com zero chuva. O pré-processador deve tratar esses casos
diferentemente.

In [ ]:
if df_inmet.empty:
    print('Dados INMET não disponíveis — pulando seção 3.')
else:
    completeness = (
        df_inmet.resample('MS').apply(lambda s: s.notna().mean())
        .assign(year=lambda d: d.index.year, month=lambda d: d.index.month)
    )

    fig, axes = plt.subplots(1, len(df_inmet.columns), figsize=(5 * len(df_inmet.columns), 8),
                              sharey=True)
    if len(df_inmet.columns) == 1:
        axes = [axes]

    month_names = ['J','F','M','A','M','J','J','A','S','O','N','D']

    for ax, station_id in zip(axes, df_inmet.columns):
        pivot = (
            df_inmet[[station_id]]
            .resample('MS').apply(lambda s: s.notna().mean())
            .assign(year=lambda d: d.index.year, month=lambda d: d.index.month)
            .pivot(index='year', columns='month', values=station_id)
        )
        pivot.columns = month_names
        sns.heatmap(pivot, ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
                    linewidths=0.3, cbar=False)
        ax.set_title(f'Est. {station_id}', fontsize=9)
        ax.set_xlabel('')

    fig.suptitle('Completude INMET por Estação (%)', fontsize=12, y=1.01)
    fig.tight_layout()
    fig.savefig(FIGDIR / '10_completude_inmet.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Estatísticas de cobertura
    print('Completude por estação:')
    print(df_inmet.notna().mean().mul(100).round(1).to_string())

## 4. Comparação INMET × CHIRPS

**Por que esta comparação é crítica?**

Se INMET e CHIRPS concordam bem, podemos usar CHIRPS para preencher
os períodos sem cobertura INMET (antes de 2000) e ter uma série
consistente de 1981 a hoje.

Se divergem sistematicamente, precisamos entender *por quê*:
- **CHIRPS superestima chuva intensa?** Comum em sistemas convectivos
  rápidos (o CHIRPS interpola e pode suavizar picos extremos)
- **INMET está em local atípico?** (vale protegido, zona urbana)
- **Sazonalidade do bias?** O bias muda entre verão e inverno?

A resposta determina se usamos CHIRPS diretamente ou aplicamos
bias correction antes de treinar o modelo.

In [ ]:
if df_inmet.empty or df_chirps.empty:
    print('Dados insuficientes para comparação — pulando seção 4.')
else:
    # Média das estações INMET como proxy da média areal observada
    inmet_mean = df_inmet.mean(axis=1).rename('INMET_mean')
    chirps_col = df_chirps.columns[0]
    chirps = df_chirps[chirps_col].rename('CHIRPS')

    # Alinhar períodos
    common = pd.concat([inmet_mean, chirps], axis=1).dropna()
    print(f'Período comum para comparação: {common.index.min().date()} → {common.index.max().date()} ({len(common)} dias)')

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Scatter
    axes[0].scatter(common['INMET_mean'], common['CHIRPS'], s=3, alpha=0.4, color='steelblue')
    lim = max(common.max())
    axes[0].plot([0, lim], [0, lim], 'k--', lw=1, label='1:1')
    axes[0].set_xlabel('INMET média (mm/dia)')
    axes[0].set_ylabel('CHIRPS (mm/dia)')
    axes[0].set_title('Scatter INMET × CHIRPS')
    r, p = stats.pearsonr(common['INMET_mean'], common['CHIRPS'])
    axes[0].text(0.05, 0.95, f'r = {r:.3f}', transform=axes[0].transAxes,
                 va='top', fontsize=10, bbox=dict(facecolor='white', alpha=0.7))
    axes[0].legend()

    # Bias mensal
    monthly_bias = (
        (common['CHIRPS'] - common['INMET_mean'])
        .groupby(common.index.month)
        .mean()
    )
    month_names = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
    axes[1].bar(month_names, monthly_bias.values, color=['salmon' if v > 0 else 'steelblue' for v in monthly_bias])
    axes[1].axhline(0, color='k', lw=0.8)
    axes[1].set_ylabel('Bias CHIRPS - INMET (mm/dia)')
    axes[1].set_title('Bias Mensal')

    # Acumulado anual
    annual = common.resample('YE').sum()
    axes[2].plot(annual.index.year, annual['INMET_mean'], marker='o', ms=4, label='INMET', color='steelblue')
    axes[2].plot(annual.index.year, annual['CHIRPS'], marker='s', ms=4, label='CHIRPS', color='darkorange')
    axes[2].set_ylabel('Precipitação anual (mm)')
    axes[2].set_title('Acumulado Anual')
    axes[2].legend()

    fig.tight_layout()
    fig.savefig(FIGDIR / '11_inmet_vs_chirps.png', dpi=150, bbox_inches='tight')
    plt.show()

    bias_overall = (common['CHIRPS'] - common['INMET_mean']).mean()
    print(f'\nBias global CHIRPS - INMET: {bias_overall:+.2f} mm/dia ({bias_overall/common["INMET_mean"].mean()*100:+.0f}%)')
    print(f'Correlação Pearson:          {r:.3f}')

## 5. Análise de Lag Chuva → Vazão

**Esta é a análise mais importante deste notebook.**

### Como funciona a cross-correlação?

Calculamos a correlação entre a série de precipitação em `t` e a
série de vazão em `t + lag` para lags de 0 a 14 dias. O lag com
maior correlação é o "tempo típico de resposta" da bacia.

**Interpretação físico-hidrológica:**
- **Lag 0–1 dias:** escoamento direto (surface runoff) — chuva intensa
  em solo saturado ou impermeável
- **Lag 2–4 dias:** escoamento subsuperficial (lateral flow)
- **Lag > 5 dias:** contribuição de aquíferos rasos (baseflow)

Para Blumenau, esperamos lag dominante de 1–3 dias dado o tamanho
da bacia (~15.000 km²) e a topografia da Serra Geral que acelera
o escoamento superficial em eventos intensos.

**Como isso define o `hindcast_length`:**  
O hindcast_length deve ser ≥ max_lag * 2 para garantir que o LSTM
veja o evento de chuva inteiro antes de fazer a previsão.  
Se o lag máximo for 5 dias → hindcast_length ≥ 10 dias.  
O config atual com 168h (7 dias) pode precisar de ajuste.

In [ ]:
# Escolher a melhor série de precipitação disponível
if not df_chirps.empty:
    P = df_chirps.iloc[:, 0].rename('P_mm')
    precip_source = 'CHIRPS'
elif not df_inmet.empty:
    P = df_inmet.mean(axis=1).rename('P_mm')
    precip_source = 'INMET (média das estações)'
else:
    print('Nenhuma série de precipitação disponível — pulando análise de lag.')
    P = None

if P is not None:
    # Alinhar séries
    df_pq = pd.concat([P, Q], axis=1).dropna()
    print(f'Período comum P×Q: {df_pq.index.min().date()} → {df_pq.index.max().date()}')
    print(f'Fonte de precipitação: {precip_source}')

    # ── Cross-correlação com scipy ─────────────────────────────────────────────
    #
    # Por que usar scipy.signal.correlate e não pd.DataFrame.corr?
    # pd.corr calcula correlação estática entre duas colunas. Para lag
    # analysis precisamos de correlação em função do deslocamento temporal,
    # o que signal.correlate faz de forma vetorizada e eficiente.
    # Normalizamos pelo número de amostras e desvios-padrão para obter
    # coeficiente de Pearson a cada lag.
    #
    max_lag = 14  # dias

    p_std = (df_pq['P_mm'] - df_pq['P_mm'].mean()) / df_pq['P_mm'].std()
    q_std = (df_pq['Q_m3s'] - df_pq['Q_m3s'].mean()) / df_pq['Q_m3s'].std()

    xcorr = signal.correlate(q_std.values, p_std.values, mode='full') / len(df_pq)
    lags = signal.correlation_lags(len(q_std), len(p_std), mode='full')

    # Filtrar lags de 0 a max_lag (P precede Q)
    mask = (lags >= 0) & (lags <= max_lag)
    lag_vals = lags[mask]
    corr_vals = xcorr[mask]

    best_lag = lag_vals[np.argmax(corr_vals)]
    best_corr = corr_vals.max()

    # ── Plot ───────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Cross-correlogram
    axes[0].bar(lag_vals, corr_vals, color='steelblue', alpha=0.8)
    axes[0].axvline(best_lag, color='crimson', lw=2, ls='--',
                    label=f'Lag máximo = {best_lag} dias (r = {best_corr:.3f})')
    axes[0].set_xlabel('Lag (dias) — P antecede Q')
    axes[0].set_ylabel('Coeficiente de Correlação')
    axes[0].set_title('Cross-Correlação P(t) × Q(t + lag)', fontsize=11)
    axes[0].legend(fontsize=9)
    axes[0].set_xticks(range(0, max_lag + 1))

    # Scatter P × Q com lag aplicado
    p_lagged = df_pq['P_mm'].shift(best_lag)
    valid = pd.concat([p_lagged, df_pq['Q_m3s']], axis=1).dropna()
    axes[1].scatter(valid['P_mm'], valid['Q_m3s'], s=3, alpha=0.3, color='steelblue')
    axes[1].set_xlabel(f'P(t - {best_lag}d) [mm/dia]')
    axes[1].set_ylabel('Q(t) [m³/s]')
    axes[1].set_yscale('log')
    axes[1].set_title(f'Scatter P-Q com Lag = {best_lag} dias')

    r2, _ = stats.pearsonr(valid['P_mm'], np.log1p(valid['Q_m3s']))
    axes[1].text(0.97, 0.05, f'r(P, logQ) = {r2:.3f}',
                 transform=axes[1].transAxes, ha='right', fontsize=9,
                 bbox=dict(facecolor='white', alpha=0.7))

    fig.tight_layout()
    fig.savefig(FIGDIR / '12_lag_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\n=== RESULTADO DO LAG ANALYSIS ===')
    print(f'Lag de máxima correlação:  {best_lag} dias')
    print(f'Correlação no lag máximo:  {best_corr:.3f}')
    print(f'\nImplicação para o config YAML:')
    recommended_hindcast = max(best_lag * 2, 7)
    print(f'  hindcast_length recomendado: ≥ {recommended_hindcast} dias ({recommended_hindcast*24} h)')

## 6. Lag Análise por Estação do Ano

A resposta hidrológica da bacia **não é constante ao longo do ano**.

- **Solo saturado (verão/outono):** pequenas chuvas geram grandes cheias
  (runoff ratio alto, lag curto)
- **Solo seco (inverno/primavera):** grande parte da chuva é absorvida
  antes de gerar escoamento (runoff ratio baixo, lag longo)

O modelo LSTM captura isso implicitamente via estado oculto (memória
do solo). Mas confirmar que o padrão existe justifica por que o modelo
precisa de hindcast longo o suficiente para "lembrar" as condições
antecedentes de umidade do solo.

In [ ]:
if P is not None and len(df_pq) > 100:
    seasons = {
        'Verão (Dez-Fev)': [12, 1, 2],
        'Outono (Mar-Mai)': [3, 4, 5],
        'Inverno (Jun-Ago)': [6, 7, 8],
        'Primavera (Set-Nov)': [9, 10, 11],
    }

    fig, axes = plt.subplots(1, 4, figsize=(17, 4), sharey=True)

    for ax, (season_name, months) in zip(axes, seasons.items()):
        subset = df_pq[df_pq.index.month.isin(months)]
        if len(subset) < 30:
            ax.set_title(f'{season_name}\n(dados insuficientes)')
            continue

        p_s = (subset['P_mm'] - subset['P_mm'].mean()) / (subset['P_mm'].std() + 1e-6)
        q_s = (subset['Q_m3s'] - subset['Q_m3s'].mean()) / (subset['Q_m3s'].std() + 1e-6)

        xcorr_s = signal.correlate(q_s.values, p_s.values, mode='full') / len(subset)
        lags_s = signal.correlation_lags(len(q_s), len(p_s), mode='full')
        mask_s = (lags_s >= 0) & (lags_s <= 14)

        best_s = lags_s[mask_s][np.argmax(xcorr_s[mask_s])]
        ax.bar(lags_s[mask_s], xcorr_s[mask_s], color='steelblue', alpha=0.8)
        ax.axvline(best_s, color='crimson', lw=2, ls='--')
        ax.set_title(f'{season_name}\nLag máx = {best_s} dias', fontsize=9)
        ax.set_xlabel('Lag (dias)')
        if ax == axes[0]:
            ax.set_ylabel('Correlação')

    fig.suptitle('Cross-Correlação P×Q por Estação do Ano', fontsize=12, y=1.02)
    fig.tight_layout()
    fig.savefig(FIGDIR / '13_lag_by_season.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Dados insuficientes para análise sazonal de lag.')

## 7. Análise de Eventos: Chuva → Cheia

Zoom nos três eventos históricos mais severos.
Para cada evento visualizamos o hidrograma dual P + Q,
permitindo identificar visualmente:
- Quando e onde começou a chuva em relação ao pico de vazão
- Se houve chuva acumulada por dias antes do pico (solo pré-saturado)
- O tempo de recessão após o pico

**Para o modelo:** eventos com pré-saturação são os mais perigosos e os
mais difíceis de prever. O LSTM precisa do hindcast para detectar a
condição antecedente; a precipitação prevista (forecast input) determina
o volume adicional.

In [ ]:
if P is None:
    print('Nenhuma série de precipitação disponível — pulando seção 7.')
else:
    WINDOW = 20  # dias antes e depois do pico

    fig, axes = plt.subplots(len(FLOOD_EVENTS), 1,
                              figsize=(14, 5.5 * len(FLOOD_EVENTS)),
                              sharex=False)

    for ax, (date_str, label) in zip(axes, FLOOD_EVENTS.items()):
        peak = pd.Timestamp(date_str)
        t0 = peak - pd.Timedelta(days=WINDOW)
        t1 = peak + pd.Timedelta(days=WINDOW)

        q_evt = Q.loc[t0:t1]
        p_evt = P.loc[t0:t1] if not P.empty else pd.Series(dtype=float)

        # Eixo principal: vazão
        color_q = 'steelblue'
        ax.plot(q_evt.index, q_evt.values, color=color_q, lw=2, label='Vazão (m³/s)')
        ax.fill_between(q_evt.index, q_evt.values, alpha=0.15, color=color_q)
        ax.axvline(peak, color='crimson', lw=1.5, ls='--', alpha=0.7, label='Pico declarado')
        ax.set_ylabel('Vazão (m³/s)', color=color_q)
        ax.tick_params(axis='y', labelcolor=color_q)

        # Eixo secundário: precipitação (barras invertidas no topo)
        if not p_evt.empty:
            ax2 = ax.twinx()
            ax2.bar(p_evt.index, p_evt.values, color='darkgreen', alpha=0.5,
                    width=0.8, label='Precipitação (mm/dia)')
            ax2.set_ylabel('Precipitação (mm/dia)', color='darkgreen')
            ax2.tick_params(axis='y', labelcolor='darkgreen')
            ax2.invert_yaxis()  # precipitação cresce para baixo (convenção meteorológica)
            ax2.legend(loc='lower right', fontsize=8)

        ax.set_title(f'{label} | {date_str}', fontsize=11, fontweight='bold')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
        ax.legend(loc='upper left', fontsize=8)

        # Estatísticas do evento
        if not p_evt.empty and not q_evt.empty:
            total_p = p_evt.sum()
            max_q = q_evt.max()
            max_p = p_evt.max()
            ax.text(0.98, 0.97,
                    f'P total: {total_p:.0f} mm\nP máx:   {max_p:.0f} mm/dia\nQ máx:   {max_q:.0f} m³/s',
                    transform=ax.transAxes, ha='right', va='top', fontsize=9,
                    bbox=dict(facecolor='white', alpha=0.8))

    fig.suptitle('Hidrógrafas e Precipitação — Eventos Históricos\nRio Itajaí-Açu / Blumenau',
                 fontsize=13, y=1.01)
    fig.tight_layout()
    fig.savefig(FIGDIR / '14_eventos_pq.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Runoff Ratio — Sazonalidade

O **runoff ratio** (Q / P) quantifica que fração da chuva vira escoamento.

- Alto no verão/outono (solo saturado, vegetação com ET alta na wet season)
- Baixo no inverno (recarga do lençol, ET baixa)

Este padrão é fundamental para o modelo: um mesmo volume de chuva
pode gerar cheias muito diferentes dependendo das condições antecedentes.
O LSTM aprende isso via o estado oculto (memória de longo prazo),
mas ajuda validar que o sinal está presente nos dados.

In [ ]:
if P is not None and len(df_pq) > 365:
    # Acumulados mensais para runoff ratio mensal
    Q_monthly = df_pq['Q_m3s'].resample('MS').mean() * 86400 / 1000  # m³/s → mm/dia (approx)
    P_monthly = df_pq['P_mm'].resample('MS').sum()

    rr = (Q_monthly / P_monthly.replace(0, np.nan)).clip(0, 3)
    rr_by_month = rr.groupby(rr.index.month).median()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    month_names = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
    colors = ['salmon' if v > rr_by_month.median() else 'steelblue' for v in rr_by_month.values]
    axes[0].bar(month_names, rr_by_month.values, color=colors, alpha=0.85)
    axes[0].axhline(rr_by_month.median(), color='k', ls='--', lw=1)
    axes[0].set_ylabel('Runoff Ratio (Q/P)')
    axes[0].set_title('Runoff Ratio Mensal Mediano', fontsize=11)
    axes[0].set_ylim(0, rr_by_month.max() * 1.3)

    # Série anual do runoff ratio
    Q_annual = df_pq['Q_m3s'].resample('YE').mean() * 86400 * 365 / 1e9  # km³/ano
    P_annual = df_pq['P_mm'].resample('YE').sum() / 1000  # m/ano
    rr_annual = (Q_annual / (P_annual * 15000)).clip(0, 3)  # /área em km²

    axes[1].plot(rr_annual.index.year, rr_annual.values, marker='o', ms=5, color='steelblue')
    axes[1].axhline(rr_annual.mean(), color='k', ls='--', lw=1, label=f'Média = {rr_annual.mean():.2f}')
    axes[1].set_ylabel('Runoff Ratio Anual')
    axes[1].set_title('Evolução Temporal do Runoff Ratio')
    axes[1].legend(fontsize=9)

    fig.tight_layout()
    fig.savefig(FIGDIR / '15_runoff_ratio.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('Runoff Ratio mediano por mês:')
    print(dict(zip(month_names, rr_by_month.values.round(2))))
else:
    print('Dados insuficientes para análise de runoff ratio.')

## 9. Resumo — Decisões para o Pipeline

Com base nas análises acima, documentamos as decisões para os próximos passos:

In [ ]:
decisions = {
    'fonte_precipitacao_treino': {
        'decisão': 'CHIRPS (1981–presente) como série principal; INMET como validação',
        'justificativa': 'CHIRPS cobre todo o período histórico, incluindo os eventos '
                         'de 1983 e 2008 sem gaps. INMET só cobre 2000–presente. '
                         'Se bias CHIRPS-INMET for > 15%, aplicar correção de bias antes.',
    },
    'hindcast_length': {
        'decisão': 'Confirmar via lag analysis — mínimo 2x lag máximo',
        'justificativa': 'Bacia de ~15.000 km² com topografia íngreme tem lag estimado '
                         'de 2–4 dias. Config atual de 168h (7 dias) deve ser suficiente, '
                         'mas lag analysis pode sugerir revisão para 14+ dias.',
    },
    'representacao_espacial': {
        'decisão': 'Média areal CHIRPS como único feature de precipitação na v1',
        'justificativa': 'O OpenHydroNet base aceita um único feature de precipitação por bacia. '
                         'Versão futura pode incluir múltiplos pixels ou features de '
                         'variabilidade espacial (std dos pixels da bacia).',
    },
    'proximo_passo': {
        'decisão': 'notebooks/02_preprocessing: rodar caravan_formatter.py e validar NC',
        'justificativa': 'Antes de treinar, validar visualmente o NetCDF: datas corretas, '
                         'streamflow em m³/s sem valores aberrantes, precipitation alinhada.',
    },
}

for key, val in decisions.items():
    print(f'\n── {key} ──')
    print(f'  Decisão:       {val["decisão"]}')
    print(f'  Justificativa: {val["justificativa"]}')